# 2. Prétraitement et extraction de features

Objectif : préparer les images pour un modèle CNN pré-entraîné, extraire des embeddings visuels avec ResNet, puis sauvegarder les vecteurs de features pour les étapes de clustering et semi-supervisé.

In [ ]:
# Importation des bibliothèques nécessaires
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from tqdm.auto import tqdm # barre de progression

c:\Users\louis\Documents_locaux\LC_10\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Configuration des chemins
DATA_DIR = Path("data/raw")
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
IMAGE_SIZE = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # si une carte graphique NVIDIA compatible CUDA est disponible : utiliser "cuda" sinon : utiliser "cpu"
device

device(type='cpu')

In [4]:
# Charger les chemins des images et leurs labels
image_paths = sorted(DATA_DIR.rglob("*.jpg"))

records = []

for path in image_paths:
    if "avec_labels" in path.parts:
        split = "avec_labels"
        label = path.parent.name
    elif "sans_label" in path.parts:
        split = "sans_label"
        label = "unknown"
    else:
        split = "unknown"
        label = "unknown"

    records.append({
        "path": str(path),
        "filename": path.name,
        "split": split,
        "label": label
    })

metadata_df = pd.DataFrame(records)

print("Nombre total d'images :", len(metadata_df))
display(metadata_df.head())
display(metadata_df["label"].value_counts())

Nombre total d'images : 1506


,path,filename,split,label
0,data\raw\avec_labels\cancer\05340cd4-3bb2-459d...,05340cd4-3bb2-459d-9937-bf27d52d8351.jpg,avec_labels,cancer
1,data\raw\avec_labels\cancer\0c6f3641-60d9-4a76...,0c6f3641-60d9-4a76-abe5-de89d55d5f2c.jpg,avec_labels,cancer
2,data\raw\avec_labels\cancer\0f718241-8f63-4b55...,0f718241-8f63-4b55-81ce-315324b51069.jpg,avec_labels,cancer
3,data\raw\avec_labels\cancer\11a7a426-4806-401e...,11a7a426-4806-401e-98b2-b96e7094d1a6.jpg,avec_labels,cancer
4,data\raw\avec_labels\cancer\1c043dbb-4623-4769...,1c043dbb-4623-4769-8e5e-0223bd745040.jpg,avec_labels,cancer


label
unknown    1406
cancer       50
normal       50
Name: count, dtype: int64

In [5]:
# Définition preprocessing ResNet
preprocess = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

Les moyennes et écarts-types utilisés correspondent à la normalisation ImageNet, car ResNet a été pré-entraîné sur ImageNet.  
Même si les images médicales sont différentes, cette normalisation permet d'utiliser correctement les poids pré-entraînés.

In [6]:
# Dataset PyTorch personnalisé
class BrainScanDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_path = row["path"]

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, row["path"], row["label"], row["split"]

In [7]:
# DataLoader
dataset = BrainScanDataset(metadata_df, transform=preprocess)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

len(dataset), len(dataloader)

(1506, 48)

In [8]:
# Charger ResNet pré-entraîné
weights = models.ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=weights)
resnet

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\louis/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [9]:
# Retirer la dernière couche (couche fully connected) pour obtenir les features
feature_extractor = torch.nn.Sequential(
    *list(resnet.children())[:-1]
)

feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

for param in feature_extractor.parameters():
    param.requires_grad = False

Ici, on garde ResNet comme extracteur de features : on enlève la dernière couche qui prédit les classes ImageNet, et on récupère le vecteur juste avant.

In [10]:
# Extraction des embeddings
all_features = []
all_paths = []
all_labels = []
all_splits = []

with torch.no_grad():
    for images, paths, labels, splits in tqdm(dataloader):
        images = images.to(device)

        features = feature_extractor(images)
        features = features.view(features.size(0), -1)

        all_features.append(features.cpu().numpy())
        all_paths.extend(paths)
        all_labels.extend(labels)
        all_splits.extend(splits)

features_array = np.vstack(all_features)

features_array.shape

100%|██████████| 48/48 [00:30<00:00,  1.58it/s]


(1506, 512)

In [11]:
# Création d'un DataFrame pour stocker les features extraites
feature_columns = [f"feature_{i}" for i in range(features_array.shape[1])]

features_df = pd.DataFrame(features_array, columns=feature_columns)

features_df.insert(0, "path", all_paths)
features_df.insert(1, "label", all_labels)
features_df.insert(2, "split", all_splits)

features_df.head()

,path,label,split,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,...,feature_502,feature_503,feature_504,feature_505,feature_506,feature_507,feature_508,feature_509,feature_510,feature_511
0,data\raw\avec_labels\cancer\05340cd4-3bb2-459d...,cancer,avec_labels,0.223682,0.710136,1.481257,0.459431,0.692085,0.185364,2.059628,...,0.113938,1.406707,0.216484,1.110025,1.869285,0.236830,0.708141,0.785015,0.015425,0.529030
1,data\raw\avec_labels\cancer\0c6f3641-60d9-4a76...,cancer,avec_labels,2.303200,1.045277,1.294456,2.547840,0.545068,0.133437,1.212342,...,0.162986,0.605513,2.760418,0.387895,1.810046,1.444754,0.330206,1.192759,0.032950,0.648934
2,data\raw\avec_labels\cancer\0f718241-8f63-4b55...,cancer,avec_labels,3.997212,1.404900,1.149433,0.966352,1.491258,0.024406,1.677062,...,0.545248,1.358088,2.565499,0.517149,2.708174,2.859923,0.957908,1.831673,0.093461,0.461350
3,data\raw\avec_labels\cancer\11a7a426-4806-401e...,cancer,avec_labels,2.034307,1.851072,1.561280,0.760116,0.673758,0.079369,3.240707,...,0.443000,0.254960,3.330342,1.184569,0.911555,1.283250,0.122348,0.274706,0.959745,1.228101
4,data\raw\avec_labels\cancer\1c043dbb-4623-4769...,cancer,avec_labels,2.823692,1.664199,1.330548,1.195984,2.489143,0.071206,2.423799,...,0.522816,0.000000,1.554175,0.503567,0.093622,0.379601,1.066818,1.798177,0.003183,1.693390


In [12]:
# Vérification des dimensions et des valeurs manquantes
print("Shape du tableau final :", features_df.shape)
print("Nombre de valeurs manquantes :", features_df.isna().sum().sum())

display(features_df[["label", "split"]].value_counts().reset_index(name="count"))
display(features_df.head())

Shape du tableau final : (1506, 515)
Nombre de valeurs manquantes : 0


,label,split,count
0,unknown,sans_label,1406
1,cancer,avec_labels,50
2,normal,avec_labels,50


,path,label,split,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,...,feature_502,feature_503,feature_504,feature_505,feature_506,feature_507,feature_508,feature_509,feature_510,feature_511
0,data\raw\avec_labels\cancer\05340cd4-3bb2-459d...,cancer,avec_labels,0.223682,0.710136,1.481257,0.459431,0.692085,0.185364,2.059628,...,0.113938,1.406707,0.216484,1.110025,1.869285,0.236830,0.708141,0.785015,0.015425,0.529030
1,data\raw\avec_labels\cancer\0c6f3641-60d9-4a76...,cancer,avec_labels,2.303200,1.045277,1.294456,2.547840,0.545068,0.133437,1.212342,...,0.162986,0.605513,2.760418,0.387895,1.810046,1.444754,0.330206,1.192759,0.032950,0.648934
2,data\raw\avec_labels\cancer\0f718241-8f63-4b55...,cancer,avec_labels,3.997212,1.404900,1.149433,0.966352,1.491258,0.024406,1.677062,...,0.545248,1.358088,2.565499,0.517149,2.708174,2.859923,0.957908,1.831673,0.093461,0.461350
3,data\raw\avec_labels\cancer\11a7a426-4806-401e...,cancer,avec_labels,2.034307,1.851072,1.561280,0.760116,0.673758,0.079369,3.240707,...,0.443000,0.254960,3.330342,1.184569,0.911555,1.283250,0.122348,0.274706,0.959745,1.228101
4,data\raw\avec_labels\cancer\1c043dbb-4623-4769...,cancer,avec_labels,2.823692,1.664199,1.330548,1.195984,2.489143,0.071206,2.423799,...,0.522816,0.000000,1.554175,0.503567,0.093622,0.379601,1.066818,1.798177,0.003183,1.693390


In [13]:
# Sauvegarde des features et du metadata
features_csv_path = OUTPUT_DIR / "resnet18_features.csv"
features_npy_path = OUTPUT_DIR / "resnet18_features.npy"
metadata_path = OUTPUT_DIR / "metadata.csv"

features_df.to_csv(features_csv_path, index=False)
np.save(features_npy_path, features_array)

metadata_df.to_csv(metadata_path, index=False)

print("Features CSV sauvegardées :", features_csv_path)
print("Features NPY sauvegardées :", features_npy_path)
print("Metadata sauvegardées :", metadata_path)

Features CSV sauvegardées : data\processed\resnet18_features.csv
Features NPY sauvegardées : data\processed\resnet18_features.npy
Metadata sauvegardées : data\processed\metadata.csv


In [14]:
# Chargement des features sauvegardées pour vérification
loaded_features_df = pd.read_csv(features_csv_path)
loaded_features_array = np.load(features_npy_path)

print("CSV :", loaded_features_df.shape)
print("NPY :", loaded_features_array.shape)

loaded_features_df.head()

CSV : (1506, 515)
NPY : (1506, 512)


,path,label,split,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,...,feature_502,feature_503,feature_504,feature_505,feature_506,feature_507,feature_508,feature_509,feature_510,feature_511
0,data\raw\avec_labels\cancer\05340cd4-3bb2-459d...,cancer,avec_labels,0.223682,0.710136,1.481257,0.459431,0.692085,0.185364,2.059628,...,0.113938,1.406707,0.216484,1.110025,1.869285,0.236830,0.708141,0.785015,0.015425,0.529030
1,data\raw\avec_labels\cancer\0c6f3641-60d9-4a76...,cancer,avec_labels,2.303200,1.045277,1.294456,2.547840,0.545068,0.133437,1.212342,...,0.162986,0.605513,2.760418,0.387895,1.810046,1.444754,0.330206,1.192759,0.032950,0.648934
2,data\raw\avec_labels\cancer\0f718241-8f63-4b55...,cancer,avec_labels,3.997212,1.404900,1.149433,0.966352,1.491258,0.024406,1.677062,...,0.545248,1.358088,2.565499,0.517149,2.708174,2.859923,0.957908,1.831673,0.093461,0.461350
3,data\raw\avec_labels\cancer\11a7a426-4806-401e...,cancer,avec_labels,2.034307,1.851072,1.561280,0.760116,0.673758,0.079369,3.240707,...,0.443000,0.254960,3.330342,1.184569,0.911555,1.283250,0.122348,0.274706,0.959745,1.228101
4,data\raw\avec_labels\cancer\1c043dbb-4623-4769...,cancer,avec_labels,2.823692,1.664199,1.330548,1.195984,2.489143,0.071206,2.423799,...,0.522816,0.000000,1.554175,0.503567,0.093622,0.379601,1.066818,1.798177,0.003183,1.693391


In [ ]:
# Vérification statistique des embeddings ResNet
# Cette cellule permet de vérifier rapidement la qualité numérique des embeddings extraits : absence de valeurs manquantes, variabilité des features et cohérence du nombre d’observations.
features_only = features_df[feature_columns]

summary = features_only.describe().T
summary.head()

,count,mean,std,min,25%,50%,75%,max
feature_0,1506.0,1.710204,1.068290,0.032112,0.759476,1.665857,2.424318,5.447840
feature_1,1506.0,0.651527,0.513970,0.000000,0.226021,0.550998,0.935759,3.318114
feature_2,1506.0,1.145683,0.620622,0.000000,0.688853,1.079283,1.561245,3.734446
feature_3,1506.0,1.263057,0.598859,0.069285,0.810025,1.209444,1.633162,3.758924
feature_4,1506.0,0.655097,0.684485,0.000000,0.147923,0.414853,0.923540,3.751265


## Observations

Les images ont été redimensionnées en 224 x 224 pixels afin d'être compatibles avec ResNet18.

Les pixels ont été normalisés avec les statistiques ImageNet, car le modèle utilisé est pré-entraîné sur ImageNet.

Les couches convolutives du modèle ont été gelées : ResNet18 est utilisé uniquement comme extracteur de features, sans ré-entraînement.

La dernière couche de classification a été retirée. Pour chaque image, on récupère un embedding visuel de dimension 512.

Le tableau final contient une ligne par image, avec :
- le chemin de l'image ;
- son label lorsqu'il existe ;
- son type de split ;
- les 512 features extraites.

Ces embeddings pourront être utilisés dans les étapes suivantes pour :
- réaliser un clustering exploratoire ;
- projeter les images en 2D ;
- entraîner des modèles semi-supervisés ;
- prédire les labels manquants.

## Definition of Done - Étape 2

L'étape 2 est considérée comme terminée si :

- les images sont chargées correctement ;
- un preprocessing adapté à ResNet est appliqué ;
- un modèle pré-entraîné est utilisé ;
- les couches du modèle sont gelées ;
- la couche de classification finale est retirée ;
- un embedding est extrait pour chaque image ;
- les features sont sauvegardées dans un format exploitable ;
- les métadonnées sont conservées avec les chemins et labels ;
- les fichiers de sortie peuvent être rechargés sans erreur.